# 1 · Bid request and the data behind it

Open the demo with the workload. A DSP receives an OpenRTB bid request,
resolves the user, matches eligible ads, and returns a bid — all inside
~10 ms. This notebook walks through the **data** Redis is holding to make
that possible. Everything that follows in notebooks 2–6 is about getting
that match-and-return done as cheaply as possible.

The local docker-compose stack is expected to be running. From the repo root:

```bash
make up
python3 -m data.load_redis --redis-url redis://localhost:6381/0 \
  --dataset-dir data/generated/synthetic
```


In [1]:
from notebooks._demo_setup import connect_redis
client = connect_redis()

connected to redis://localhost:6381/0
  users=4000  campaigns=2500  precompute_version=v17_2500_12


## A single MAID profile

`maid:{maid_id}` is a Redis HASH carrying the cardholder's full profile.
At the customer scale this is up to 500 M entries with up to 500 taxonomy
labels and 50 publisher-hashed identities each; the synthetic dataset uses
a coarsened version of the same shape so the mechanics are identical.


In [2]:
raw = client.hgetall('maid:maid_00042')
print('hash field count:', len(raw))
print('sample fields:')
for key in ['user_id', 'geo', 'state', 'device', 'card_tier']:
    print(f'  {key:>18} = {raw[key]!r}')
print(f'  {"interests_json":>18} = {raw["interests_json"][:120]} ...')

hash field count: 14
sample fields:
             user_id = 'maid_00042'
                 geo = 'US'
               state = 'IL'
              device = 'Roku'
           card_tier = 'Standard'
      interests_json = {"camping": 0.3801, "family": 0.2108, "finance": 0.6832, "fitness": 0.8003, "foodie": 0.3884, "gaming": 0.615, "home_imp ...


## The hot scoring profile

`maid_hot:{maid_id}` is a stripped-down hash carrying only the fields the
*online* path needs. Most of the cold profile (state, postal_code, full
identity list) is left in `maid:` for offline jobs and audit. This split
matters once the dataset is at production scale — the bid path only ever
reads the hot subset, which fits in a single `HMGET` round trip.


In [3]:
hot = client.hmget('maid_hot:maid_00042', 'user_id', 'interests_json', 'impression_count')
print('user_id          =', hot[0])
print('impression_count =', hot[2])
print('interests_json   =', hot[1][:160], '...')

user_id          = maid_00042
impression_count = 233
interests_json   = {"camping": 0.3801, "family": 0.2108, "finance": 0.6832, "fitness": 0.8003, "foodie": 0.3884, "gaming": 0.615, "home_improvement": 0.1571, "luxury": 0.18, "pet_ ...


## Identity resolution

The publisher hands the bid engine a tokenised identifier. Bid time, the
engine resolves it to a `maid_id` via a single `GET` against an `identity:`
keyspace populated alongside the MAIDs.


In [4]:
sample_token = 'id_00042_01'
maid_id = client.get(f'identity:{sample_token}')
print(f'identity:{sample_token}  ->  {maid_id}')

identity:id_00042_01  ->  maid_00042


## Active ad cache

`campaign:{id}` and `campaign_state:{id}` together carry an ad's static
targeting and its mutable delivery state (pacing, budget, frequency cap).
Static fields update every few minutes from the ad-platform feed; mutable
state updates in real time as wins land.


In [5]:
campaign = client.hgetall('campaign:c00042')
state = client.hgetall('campaign_state:c00042')

print('campaign:c00042 (static):')
for key in ['campaign_id', 'geo_json', 'device_json', 'card_tiers_json',
            'required_segments_json', 'any_of_segments_json',
            'taxonomy_filter_json', 'bid']:
    if key in campaign:
        print(f'  {key:>22} = {campaign[key][:80]}')
print()
print('campaign_state:c00042 (mutable):')
for key, value in state.items():
    print(f'  {key:>22} = {value}')

campaign:c00042 (static):
             campaign_id = c00042
                geo_json = ["*"]
             device_json = ["Web"]
         card_tiers_json = ["*"]
  required_segments_json = ["fitness_high", "camping_high"]
    any_of_segments_json = []
    taxonomy_filter_json = {"and": [{"gte": ["tech", 0.55]}, {"gte": ["foodie", 0.35]}, {"not": {"or": [{"g
                     bid = 3.724

campaign_state:c00042 (mutable):
             campaign_id = c00042
           pacing_status = active
        daily_budget_usd = 8933.91
         spent_today_usd = 2679.2
           frequency_cap = 2
                  status = active


## What the bid path needs from this data

For a single bid, the engine needs to:

1. resolve `identity_token` → `maid_id` (one Redis op),
2. fetch the MAID's scoring fields (one Redis op),
3. filter the ~5 K active ads against the MAID's static profile, mutable
   state, and the per-ad `taxonomy_filter`,
4. score the survivors and return the top few.

The next five notebooks are seven different ways to do step 3, in order
from naive to fast.


In [6]:
# Brief sanity check: do we have all the keyspaces we'll need across the demo?
for prefix in ['maid:', 'maid_hot:', 'identity:', 'aud:', 'campaign:',
                'campaign_state:', 'idx:', 'fcap:', 'bm:']:
    cursor, keys = client.scan(cursor=0, match=f'{prefix}*', count=200)
    print(f'  {prefix:<18} sample: {keys[:2]}')

  maid:              sample: ['maid:maid_00988', 'maid:maid_03663']
  maid_hot:          sample: ['maid_hot:maid_02361', 'maid_hot:maid_02670']
  identity:          sample: ['identity:id_02570_04', 'identity:id_00420_01']
  aud:               sample: ['aud:maid_01342', 'aud:maid_03201']
  campaign:          sample: ['campaign:c01923', 'campaign:c00039']
  campaign_state:    sample: ['campaign_state:c01265', 'campaign_state:c01559']
  idx:               sample: ['idx:geo:US']
  fcap:              sample: ['fcap:maid_03553', 'fcap:maid_00859']
  bm:                sample: []
